# 1. 설치 검증

In [1]:
import rustima
import numpy as np

print(rustima.version())                              # "0.1.0"
y = np.random.randn(100).cumsum()
r = rustima.sarimax_fit(y, order=(1, 1, 1), seasonal=(0, 0, 0, 0))
print(f"converged={r['converged']}, AIC={r['aic']:.2f}")

0.1.0
converged=True, AIC=303.92


# 2. 예제

## 2.1. 첫 예측(5분 워크스루)
- converged=True → 최적화 성공
- 낮은 AIC / BIC → 더 나은 모델 적합 (다른 차수 대비 상대적으로)
- Ljung-Box p > 0.05 → 잔차가 백색잡음처럼 보임 (good)
- ci_lower / ci_upper → 불확실성 밴드; 넓을수록 덜 확신

In [ ]:
import numpy as np
from rustima import SARIMAXModel, auto_arima

# ── 1. 트렌드 + 1년 계절성이 있는 월별 매출 데이터 시뮬레이션 ────────────
rng = np.random.default_rng(42)
n = 120  # 10년치 월별
trend = 0.5 * np.arange(n)                           # 선형 상승 트렌드
season = 10 * np.sin(2 * np.pi * np.arange(n) / 12)  # 1년 주기 (s=12)
noise = rng.normal(0, 1.0, n)
y = trend + season + noise

# ── 2. auto_arima가 차수를 알아서 선택 ────────────────────────────────────
auto_result = auto_arima(y, s=12, trace=True)  # trace=True → 시도한 모델 출력
print(auto_result.search_summary())
# >>> Best: SARIMA(0,1,1)(0,1,1)[12]  AIC=345.67  (evaluated 23 models)

# ── 3. 선택된 모델 살펴보기 ────────────────────────────────────────────────
model = auto_result.result              # SARIMAXResult 객체
print(model.summary())                  # statsmodels 스타일 파라미터 테이블
print(f"AIC={model.aic:.2f}  BIC={model.bic:.2f}")

# ── 4. 다음 12개월 95% 신뢰구간으로 예측 ──────────────────────────────────
forecast = model.forecast(steps=12, alpha=0.05)
df = forecast.to_dataframe()            # Polars DataFrame
print(df)
# 형태 (12, 5): step | mean | variance | ci_lower | ci_upper

# ── 5. 잔차 검사 (랜덤 노이즈처럼 보여야 함) ──────────────────────────────
diag = model.diagnostics()
print(f"Ljung-Box p-value: {diag['ljung_box_pvalue']:.3f}  (>0.05 이면 good)")

  ARIMA(0,1,0)(0,1,0)[12] : aic=395.332
  ARIMA(2,1,2)(0,1,0)[12] : aic=328.028
  ARIMA(1,1,0)(0,1,0)[12] : aic=361.325
  ARIMA(0,1,1)(0,1,0)[12] : aic=322.248
  ARIMA(0,1,0)(1,1,0)[12] : aic=376.731
  ARIMA(0,1,0)(0,1,1)[12] : aic=369.145
  ARIMA(1,1,0)(1,1,0)[12] : aic=337.221
  ARIMA(0,1,1)(0,1,1)[12] : aic=294.530
  ARIMA(1,1,1)(0,1,1)[12] : aic=296.545
  ARIMA(0,1,2)(0,1,1)[12] : aic=296.574
  ARIMA(0,1,1)(1,1,1)[12] : aic=294.959
  ARIMA(0,1,1)(0,1,2)[12] : aic=295.058
  ARIMA(1,1,2)(0,1,1)[12] : aic=297.474
auto_arima: Best ARIMA(0,1,1)(0,1,1)[12]
  aic=294.530
  Models evaluated: 13 (13 converged)
                               SARIMAX Results                                
Model: SARIMAX(0,1,1)(0,1,1)[12]                  Log Likelihood:     -144.265
No. Observations: 120                              AIC:                294.530
Trend: n                                           BIC:                302.892
Method: lbfgsb                                     HQIC:               